In [ ]:
import pandas as pd
import requests
import time
from io import StringIO
from pybaseball import statcast, chadwick_register

# ============================================================
# 공통 설정
# ============================================================
TEAM_ID   = 140
TEAM_CODE = "TEX"
SEASON    = 2025
START     = "2025-03-27"
END       = "2025-09-28"
SAVE_DIR  = "./data"
HEADERS   = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
MONTHS    = ['March','April','May','June','July','August','September','October']
POS_MAP   = {'P':1,'C':2,'1B':3,'2B':4,'3B':5,'SS':6,'LF':7,'CF':8,'RF':9,'DH':10}


In [ ]:
# ============================================================
# Statcast pitch-level 수집 + 이름 매핑
# ============================================================

def fetch_statcast(start_date, end_date, team_code):
    """Statcast 수집 → 팀 필터 + 정규시즌만 반환"""
    dt = statcast(start_dt=start_date, end_dt=end_date)
    dt = dt[(dt["home_team"] == team_code) | (dt["away_team"] == team_code)]
    dt = dt[dt["game_type"] == "R"]
    return dt.convert_dtypes(dtype_backend="numpy_nullable")


def load_player_names():
    """Chadwick register → {mlbam_id: player_name}"""
    reg = chadwick_register(save=True)
    reg = reg[["key_mlbam","name_first","name_last"]].dropna(subset=["key_mlbam"])
    reg["key_mlbam"]    = reg["key_mlbam"].astype(int)
    reg["player_name"]  = (reg["name_first"].fillna("") + " " + reg["name_last"].fillna("")).str.strip()
    return dict(zip(reg["key_mlbam"], reg["player_name"]))


def _add_rate_cols(df, bip_col, cols):
    """비율 지표 일괄 계산: cols = [(numerator, new_col_name), ...]"""
    for num_col, new_col in cols:
        df[new_col] = (df[num_col] / df[bip_col].replace(0, 1)) * 100
    return df


def aggregate_batters(df, id_to_name):
    """타자 일별 Statcast 집계"""
    bat_rows = df[
        ((df["home_team"] == TEAM_CODE) & (df["inning_topbot"] == "Bot")) |
        ((df["away_team"] == TEAM_CODE) & (df["inning_topbot"] == "Top"))
    ]
    agg = bat_rows.groupby(["batter","game_date"]).agg(
        PA            = ("at_bat_number",                "nunique"),
        Pitches_seen  = ("pitch_type",                  "count"),
        EV            = ("launch_speed",                 "mean"),
        Max_EV        = ("launch_speed",                 "max"),
        LA            = ("launch_angle",                 "mean"),
        Barrel        = ("launch_speed_angle",           lambda x: (x == 6).sum()),
        Hard_Hit      = ("launch_speed",                 lambda x: (x >= 95).sum()),
        Sweet_Spot    = ("launch_angle",                 lambda x: ((x >= 8) & (x <= 32)).sum()),
        Total_BIP     = ("launch_speed",                 "count"),
        xBA           = ("estimated_ba_using_speedangle","mean"),
        xSLG          = ("estimated_slg_using_speedangle","mean"),
        xwOBA         = ("estimated_woba_using_speedangle","mean"),
        Whiff         = ("description",                  lambda x: (x == "swinging_strike").sum()),
        Swing         = ("description",                  lambda x: x.isin(["swinging_strike","hit_into_play","foul","swinging_strike_blocked"]).sum()),
        Bat_Speed     = ("bat_speed",                    "mean"),
        Swing_Length  = ("swing_length",                 "mean"),
        Attack_Angle  = ("attack_angle",                 "mean"),
    ).reset_index()

    agg.insert(1, "player_name", agg["batter"].map(id_to_name))
    _add_rate_cols(agg, "Total_BIP", [
        ("Barrel",    "Barrel_pct"),
        ("Hard_Hit",  "Hard_Hit_pct"),
        ("Sweet_Spot","Sweet_Spot_pct"),
    ])
    agg["Whiff_pct"] = (agg["Whiff"] / agg["Swing"].replace(0,1)) * 100
    return agg


def aggregate_pitchers(df, id_to_name):
    """투수 일별 Statcast 집계"""
    pit_rows = df[
        ((df["home_team"] == TEAM_CODE) & (df["inning_topbot"] == "Top")) |
        ((df["away_team"] == TEAM_CODE) & (df["inning_topbot"] == "Bot"))
    ]
    agg = pit_rows.groupby(["pitcher","game_date"]).agg(
        Pitches_thrown      = ("pitch_type",                   "count"),
        Avg_Velo            = ("release_speed",                "mean"),
        Max_Velo            = ("release_speed",                "max"),
        Avg_Spin            = ("release_spin_rate",            "mean"),
        EV_allowed          = ("launch_speed",                 "mean"),
        Barrel_allowed      = ("launch_speed_angle",           lambda x: (x == 6).sum()),
        Hard_Hit_allowed    = ("launch_speed",                 lambda x: (x >= 95).sum()),
        Total_BIP           = ("launch_speed",                 "count"),
        xBA_against         = ("estimated_ba_using_speedangle","mean"),
        xwOBA_against       = ("estimated_woba_using_speedangle","mean"),
        Whiff               = ("description",                  lambda x: (x == "swinging_strike").sum()),
        Swing               = ("description",                  lambda x: x.isin(["swinging_strike","hit_into_play","foul","swinging_strike_blocked"]).sum()),
        K                   = ("events",                       lambda x: (x == "strikeout").sum()),
        BB                  = ("events",                       lambda x: (x == "walk").sum()),
        Outs                = ("events",                       lambda x: x.isin(["strikeout","field_out","force_out","grounded_into_double_play","double_play","sac_fly","sac_bunt","fielders_choice_out"]).sum()),
        V_Break             = ("api_break_z_with_gravity",     "mean"),
        H_Break             = ("api_break_x_arm",              "mean"),
        Release_Extension   = ("release_extension",            "mean"),
    ).reset_index()

    agg.insert(1, "player_name", agg["pitcher"].map(id_to_name))
    agg["IP_float"]             = agg["Outs"] / 3
    agg["Whiff_pct"]            = (agg["Whiff"] / agg["Swing"].replace(0,1)) * 100
    agg["Hard_Hit_allowed_pct"] = (agg["Hard_Hit_allowed"] / agg["Total_BIP"].replace(0,1)) * 100
    return agg


In [ ]:
# ============================================================
# 로스터 수집 + Savant 팀 스탯 스크래핑
# ============================================================

def fetch_roster(team_id, season):
    """MLB Stats API → {name: {id, pos}} + batters/pitchers 분리"""
    url  = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/roster?season={season}"
    data = requests.get(url).json()
    roster = {
        p['person']['fullName']: {
            'id':  str(p['person']['id']),
            'pos': p['position']['abbreviation']
        }
        for p in data['roster']
    }
    batters  = {n: i for n, i in roster.items() if i['pos'] != 'P'}
    pitchers = {n: i for n, i in roster.items() if i['pos'] == 'P'}
    return roster, batters, pitchers


def build_roster_df(roster):
    """roster dict → DataFrame"""
    return pd.DataFrame([
        {'player_id': info['id'], 'name': name,
         'pos': info['pos'],
         'type': 'pitcher' if info['pos'] == 'P' else 'batter'}
        for name, info in roster.items()
    ])


def fetch_savant_team_stats(team_id, season):
    """
    Baseball Savant 팀 페이지 테이블 스크래핑
    returns: dict of DataFrames
    """
    def _scrape(nav):
        url = f"https://baseballsavant.mlb.com/team/{team_id}?view=statcast&nav={nav}&season={season}"
        html = requests.get(url, headers=HEADERS).text
        return pd.read_html(StringIO(html))

    hit_tables = _scrape("hitting")
    pit_tables = _scrape("pitching")

    def _clean(t, level=1):
        df = t.copy()
        df.columns = df.columns.get_level_values(level)
        return df

    return {
        "hitting":            _clean(hit_tables[0]),
        "hitting_discipline": hit_tables[1],
        "hitting_batted":     hit_tables[2],
        "pitching":           _clean(pit_tables[0]),
        "pitching_discipline":pit_tables[1],
        "pitching_batted":    pit_tables[2],
    }


In [ ]:
# ============================================================
# 선수 게임로그 수집 (타자/투수 공통)
# ============================================================

def fetch_gamelogs(player_dict, season, pos_type='batter'):
    """
    player_dict: {name: {id, pos}}
    pos_type: 'batter' | 'pitcher'
    returns: DataFrame (전체 합친)
    """
    results = {}
    for name, info in player_dict.items():
        player_id = info['id']
        pos       = info['pos']
        pos_num   = 1 if pos_type == 'pitcher' else POS_MAP.get(pos, 9)

        url = (f"https://baseballsavant.mlb.com/player-services/stats"
               f"?statType=gamelogs&pos={pos_num}&leagueType=mlb"
               f"&playerId={player_id}&season={season}&gameType=R")
        res = requests.get(url, headers=HEADERS)

        if not res.text.strip():
            print(f"⚠️  {name} — 응답 없음, 스킵")
            continue
        try:
            data = res.json()
            if not data.get('gameLogTable'):
                print(f"⚠️  {name} — 데이터 없음, 스킵")
                continue

            df = pd.read_html(StringIO(data['gameLogTable']))[0]
            df = df[~df['Date'].isin(MONTHS)].reset_index(drop=True)
            df.insert(0, 'player_id', player_id)
            df.insert(1, 'name',      name)
            if pos_type == 'batter':
                df.insert(2, 'pos', pos)

            results[player_id] = df
            print(f"✅  {name:<22} {len(df)}경기")
        except Exception as e:
            print(f"❌  {name} — {e}")

        time.sleep(0.5)

    return pd.concat(results.values(), ignore_index=True) if results else pd.DataFrame()


In [ ]:
# ============================================================
# 전체 수집 파이프라인
# ============================================================

def collect_all(team_id=TEAM_ID, team_code=TEAM_CODE,
                season=SEASON, start=START, end=END,
                save_dir=SAVE_DIR):

    import os
    os.makedirs(save_dir, exist_ok=True)

    print("─── 1. Statcast 수집 ────────────────────")
    df         = fetch_statcast(start, end, team_code)
    id_to_name = load_player_names()

    bat_daily  = aggregate_batters(df, id_to_name)
    pit_daily  = aggregate_pitchers(df, id_to_name)
    bat_daily.to_csv(f"{save_dir}/rangers_{season}_batters_daily_final.csv",  index=False)
    pit_daily.to_csv(f"{save_dir}/rangers_{season}_pitchers_daily_final.csv", index=False)
    print(f"  타자 일별: {bat_daily.shape}  투수 일별: {pit_daily.shape}")

    print("─── 2. 로스터 수집 ──────────────────────")
    roster, batters, pitchers = fetch_roster(team_id, season)
    roster_df = build_roster_df(roster)
    roster_df.to_csv(f"{save_dir}/rangers_roster_{season}.csv", index=False)
    print(f"  타자 {len(batters)}명  투수 {len(pitchers)}명")

    print("─── 3. Savant 팀 스탯 ───────────────────")
    tables = fetch_savant_team_stats(team_id, season)
    for name, tbl in tables.items():
        tbl.to_csv(f"{save_dir}/rangers_{name}.csv", index=False)
        print(f"  {name}: {tbl.shape}")

    print("─── 4. 타자 게임로그 ─────────────────────")
    batter_gl = fetch_gamelogs(batters, season, pos_type='batter')
    batter_gl.to_csv(f"{save_dir}/rangers_batter_gamelogs.csv", index=False)

    print("─── 5. 투수 게임로그 ─────────────────────")
    pitcher_gl = fetch_gamelogs(pitchers, season, pos_type='pitcher')
    pitcher_gl.to_csv(f"{save_dir}/rangers_pitcher_gamelogs.csv", index=False)

    print("\n✅  전체 수집 완료")
    return {
        "bat_daily":  bat_daily,
        "pit_daily":  pit_daily,
        "roster":     roster_df,
        "batter_gl":  batter_gl,
        "pitcher_gl": pitcher_gl,
        **tables,
    }


# ── 실행 ──────────────────────────────────────────────────────
# collect_all()   ← 전체 실행
# 개별 실행 예시:
# df         = fetch_statcast(START, END, TEAM_CODE)
# id_to_name = load_player_names()
# bat_daily  = aggregate_batters(df, id_to_name)
